# 002 PDF To MinIO And Parse

这是 RAG 知识库学习线的第二课。

本课使用真实样例 PDF：

```text
raw/北京市密云水库防御洪水方案.pdf
```

学习目标：

1. 读取 RAG 教学环境配置。
2. 检查样例 PDF 文件是否存在。
3. 连接 MinIO，并准备教学 bucket。
4. 把原始 PDF 上传到 MinIO。
5. 使用 PyMuPDF 解析 PDF 页面文本。
6. 生成 `pages.json`，并上传到 MinIO。

本课完成后，我们会得到两类产物：

```text
MinIO: raw/{doc_id}.pdf
MinIO: parsed/{doc_id}/pages.json
Local: notebooks/rag/generated/{doc_id}/pages.json
```

## 1. 本课的位置

上一课我们讲的是整体架构。这一课开始进入知识转化流水线。

当前阶段：

```text
PDF
-> MinIO 保存原文
-> PDF 解析为 pages
```

还没有进入：

```text
chunk 切片
embedding
ES 入库
三元组抽取
Neo4j 入库
```

这些会放在后续课程。

## 2. 导入依赖

本课需要两个新依赖：

```text
minio   -> 连接对象存储
pymupdf -> 解析 PDF 文本，导入名是 fitz
```

如果这里导入失败，先在项目根目录执行：

```bash
uv pip install -r requirements.txt
```

In [2]:
import importlib.metadata
import json
import os
from hashlib import sha1
from pathlib import Path
from pprint import pprint

import fitz
from dotenv import load_dotenv
from minio import Minio
from minio.error import S3Error

print('minio', importlib.metadata.version('minio'))
print('pymupdf', importlib.metadata.version('pymupdf'))

minio 7.2.20
pymupdf 1.27.2.3


## 3. 加载项目配置

后续 notebook 统一从 `.env` 读取配置。

为了让教学样例能直接运行，这里给出当前教学环境的默认值。真实项目里不要把账号密码写死在代码中。

In [2]:
def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / 'requirements.txt').exists() and (path / 'notebooks').exists():
            return path
    return current

PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / '.env', override=False)

RAG_CONFIG = {
    'minio_endpoint': os.getenv('RAG_MINIO_ENDPOINT', '192.168.102.19:9001'),
    'minio_access_key': os.getenv('RAG_MINIO_ACCESS_KEY', 'minioadmin'),
    'minio_secret_key': os.getenv('RAG_MINIO_SECRET_KEY', 'minioadmin'),
    'minio_secure': os.getenv('RAG_MINIO_SECURE', 'false').lower() == 'true',
    'minio_bucket': os.getenv('RAG_MINIO_BUCKET', 'rag-documents'),
}

safe_config = dict(RAG_CONFIG)
safe_config['minio_secret_key'] = '***'
pprint(safe_config)
print('project_root:', PROJECT_ROOT)

{'minio_access_key': 'minioadmin',
 'minio_bucket': 'rag-documents',
 'minio_endpoint': '192.168.102.19:9001',
 'minio_secret_key': '***',
 'minio_secure': False}
project_root: /home/dev/bxc/fastapi-study


## 4. 检查样例 PDF

教学样例文件位于：

```text
raw/北京市密云水库防御洪水方案.pdf
```

我们先生成一个稳定的 `doc_id`。后续 MinIO、ES、Neo4j 都会使用这个 `doc_id` 关联同一份文档。

In [3]:
SAMPLE_PDF = PROJECT_ROOT / 'raw' / '北京市密云水库防御洪水方案.pdf'

if not SAMPLE_PDF.exists():
    raise FileNotFoundError(f'样例 PDF 不存在: {SAMPLE_PDF}')

file_bytes = SAMPLE_PDF.read_bytes()
doc_id = sha1(file_bytes).hexdigest()[:16]
file_name = SAMPLE_PDF.name

print('sample_pdf:', SAMPLE_PDF)
print('file_name:', file_name)
print('doc_id:', doc_id)
print('size MB:', round(len(file_bytes) / 1024 / 1024, 2))

sample_pdf: /home/dev/bxc/fastapi-study/raw/北京市密云水库防御洪水方案.pdf
file_name: 北京市密云水库防御洪水方案.pdf
doc_id: 63b7d4d0675426b5
size MB: 5.24


## 5. 连接 MinIO

MinIO 在这里相当于文件仓库。

第一步只做三件事：

```text
1. 连接 MinIO
2. 确认 bucket 是否存在
3. 不存在则创建 bucket
```

In [4]:
def create_minio_client(config: dict) -> Minio:
    return Minio(
        endpoint=config['minio_endpoint'],
        access_key=config['minio_access_key'],
        secret_key=config['minio_secret_key'],
        secure=config['minio_secure'],
    )

minio_client = create_minio_client(RAG_CONFIG)
bucket_name = RAG_CONFIG['minio_bucket']

if not minio_client.bucket_exists(bucket_name):
    minio_client.make_bucket(bucket_name)
    print('created bucket:', bucket_name)
else:
    print('bucket exists:', bucket_name)

bucket exists: rag-documents


## 6. 上传原始 PDF

对象路径建议保持稳定：

```text
raw/{doc_id}.pdf
```

这样即使原始文件名包含中文，MinIO 对象路径也稳定、简洁。

原始文件名会保留在后续 JSON 元数据里。

In [5]:
raw_object_name = f'raw/{doc_id}.pdf'

result = minio_client.fput_object(
    bucket_name=bucket_name,
    object_name=raw_object_name,
    file_path=str(SAMPLE_PDF),
    content_type='application/pdf',
)

print('uploaded object:', result.object_name)
print('etag:', result.etag)

stat = minio_client.stat_object(bucket_name, raw_object_name)
print('object size MB:', round(stat.size / 1024 / 1024, 2))
print('content type:', stat.content_type)

uploaded object: raw/63b7d4d0675426b5.pdf
etag: f5fcac321ba33c2c05dc98cef7630696-2
object size MB: 5.24
content type: application/pdf


## 7. 用 PyMuPDF 解析 PDF 页面

PyMuPDF 的导入名是 `fitz`。

本课先使用最简单的文本抽取方式：

```python
page.get_text('text', sort=True)
```

`sort=True` 会尽量按照页面上的阅读顺序返回文本。

In [6]:
def parse_pdf_pages(pdf_path: Path, doc_id: str, file_name: str) -> list[dict]:
    pages = []
    with fitz.open(pdf_path) as pdf:
        for page_index in range(pdf.page_count):
            page = pdf.load_page(page_index)
            text = page.get_text('text', sort=True).strip()
            pages.append(
                {
                    'doc_id': doc_id,
                    'file_name': file_name,
                    'page_no': page_index + 1,
                    'text': text,
                    'char_count': len(text),
                }
            )
    return pages

pages = parse_pdf_pages(SAMPLE_PDF, doc_id=doc_id, file_name=file_name)

print('page_count:', len(pages))
print('non_empty_pages:', sum(1 for page in pages if page['text']))
print('empty_pages:', sum(1 for page in pages if not page['text']))
print('total_chars:', sum(page['char_count'] for page in pages))

page_count: 122
non_empty_pages: 120
empty_pages: 2
total_chars: 99363


## 8. 观察解析结果

解析 PDF 时，不要只看“有没有报错”。还要看：

```text
页数是否符合预期
空白页数量是否合理
前几页文本是否像正常文本
是否出现大量乱码
```

这一步是后续切片质量的基础。

In [7]:
for page in pages[:5]:
    preview = page["text"][:180].replace("\n", " ")
    print("=" * 80)
    print("page:", page["page_no"], "chars:", page["char_count"])
    print(preview)


page: 1 chars: 20
附件4                〇
page: 2 chars: 0

page: 3 chars: 178
目 录  2024 年北京市密云水库洪水调度方案................................................................ 1  2024 年北京市密云水库防洪抢险预案................................................................ 63
page: 4 chars: 0

page: 5 chars: 1
1


## 9. 保存 pages.json

我们会把解析结果保存两份：

```text
Local: notebooks/rag/generated/{doc_id}/pages.json
MinIO: parsed/{doc_id}/pages.json
```

本地文件方便调试，MinIO 文件方便后续重建和跨 notebook 复用。

In [8]:
generated_dir = PROJECT_ROOT / 'notebooks' / 'rag' / 'generated' / doc_id
generated_dir.mkdir(parents=True, exist_ok=True)

pages_json_path = generated_dir / 'pages.json'
pages_json_path.write_text(json.dumps(pages, ensure_ascii=False, indent=2), encoding='utf-8')

print('local pages json:', pages_json_path)
print('size KB:', round(pages_json_path.stat().st_size / 1024, 2))

local pages json: /home/dev/bxc/fastapi-study/notebooks/rag/generated/63b7d4d0675426b5/pages.json
size KB: 188.93


## 10. 上传 pages.json 到 MinIO

中间产物对象路径：

```text
parsed/{doc_id}/pages.json
```

下一课切片时，可以选择从本地 JSON 读取，也可以从 MinIO 下载后读取。

In [9]:
parsed_object_name = f'parsed/{doc_id}/pages.json'

result = minio_client.fput_object(
    bucket_name=bucket_name,
    object_name=parsed_object_name,
    file_path=str(pages_json_path),
    content_type='application/json',
)

print('uploaded object:', result.object_name)
print('etag:', result.etag)

stat = minio_client.stat_object(bucket_name, parsed_object_name)
print('object size KB:', round(stat.size / 1024, 2))
print('content type:', stat.content_type)

uploaded object: parsed/63b7d4d0675426b5/pages.json
etag: 3ea52f2fb6808e45666e33f58326989a
object size KB: 188.93
content type: application/json


## 11. 本课小结

本课完成了知识转化的第一段：

```text
PDF -> MinIO raw object -> pages.json
```

当前已经有三个关键变量：

```text
doc_id
raw_object_name
parsed_object_name
```

后续课程会继续使用这些标识。

下一课要做的是：

```text
pages.json
-> 文本清洗
-> chunk 切片
-> chunks.json
```

到下一课为止，我们仍然不会进入 ES 和 Neo4j。先把文本切片质量做好，再谈检索和图谱。

## 12. 练习

请你观察本课输出后回答：

1. 这份 PDF 有多少页？有多少空白页？
2. 为什么原始 PDF 和 `pages.json` 都要保存到 MinIO？
3. 为什么 `doc_id` 要稳定，而不是每次随机生成？
4. 如果某一页 `char_count` 是 0，可能有哪些原因？